In [ ]:
from data_processing import DataSetup, FigureFormat

flight_matrix_path = 'AWW_and_ICU/global_model/pgfgleam_code/datasets/country_flow_202001 1.csv'
country_codes_path = 'AWW_and_ICU/global_model/pgfgleam_code/datasets/geom_countries_codes.csv'
population_path = 'AWW_and_ICU/global_model/pgfgleam_code/datasets/WPP2019_TotalPopulation2020.csv'

flight_matrix, normalized_flow_matrix, source_countries, target_countries, source_codes, target_codes, population, name_mapping = DataSetup(flight_matrix_path, population_path, country_codes_path).load_and_process_data().values()
FigureFormat().set_figure_format()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def simulate_independent_sliding_window(mu_trajectory, p, alpha, num_trials=500000, retest_delay=3, test_freq=1):
    """
    Tests only happen on specific days determined by test_freq.
    
    Testing schedule: Samples taken on days 0, test_freq, 2*test_freq, ...
    Results arrive retest_delay days after sampling.
    
    For n consecutive positives starting with sample at day t:
    - 1st positive result: t + retest_delay
    - 2nd positive result: t + test_freq + retest_delay
    - 3rd positive result: t + 2*test_freq + retest_delay
    """
    results = {
        1: {"is_true": [], "confirm_day": []},
        2: {"is_true": [], "confirm_day": []},
        3: {"is_true": [], "confirm_day": []},
        4: {"is_true": [], "confirm_day": []}
    }

    for _ in range(num_trials):
        found_stages = set()
        
        # Only try starting days that align with testing schedule
        possible_start_days = range(0, len(mu_trajectory), test_freq)
        
        for t in possible_start_days:
            
            # Check for each N-consecutive requirement
            for n_conseq in [1, 2, 3, 4]:
                if n_conseq in found_stages:
                    continue
                
                # Sampling days: t, t+test_freq, t+2*test_freq, ...
                sampling_days = [t + (i * test_freq) for i in range(n_conseq)]
                
                if sampling_days[-1] >= len(mu_trajectory):
                    continue
                
                # Execute the tests
                all_pos = True
                sequence_was_tp = []
                
                for sample_day in sampling_days:
                    n_particles = np.random.poisson(mu_trajectory[sample_day])
                    prob_tp = 1 - (1 - p)**n_particles
                    
                    hit_tp = np.random.rand() < prob_tp
                    hit_fp = not hit_tp and (np.random.rand() < alpha)
                    
                    if hit_tp or hit_fp:
                        sequence_was_tp.append(hit_tp)
                    else:
                        all_pos = False
                        break
                
                if all_pos:
                    # Sequence found! 
                    results[n_conseq]["is_true"].append(any(sequence_was_tp))
                    # Confirmation day = last sampling day + retest_delay
                    results[n_conseq]["confirm_day"].append(sampling_days[-1] + retest_delay)
                    found_stages.add(n_conseq)
            
            if len(found_stages) == 4:
                break
                
    return results

def plot_split_histogram(ax, detection_days, is_true_positive, bins, color, label, xlim):
    """Plots TP (colored) and FP (white) split bars."""
    days = np.array(detection_days)
    truth = np.array(is_true_positive)
    
    counts, bin_edges = np.histogram(days, bins=bins)
    tp_counts = np.histogram(days[truth], bins=bins)[0]
    
    bin_width = bin_edges[1] - bin_edges[0]
    total_area = np.sum(counts) * bin_width
    if total_area == 0: return Patch(), Patch()
    
    tp_density = tp_counts / total_area
    fp_density = (counts - tp_counts) / total_area
    
    for i in range(len(bin_edges) - 1):
        bin_center = (bin_edges[i] + bin_edges[i + 1]) / 2
        if (tp_density[i] + fp_density[i]) > 0:
            ax.bar(bin_center, tp_density[i], width=bin_width*0.8, color=color, alpha=0.7, edgecolor=color)
            ax.bar(bin_center, fp_density[i], width=bin_width*0.8, bottom=tp_density[i], color='white', edgecolor=color, linewidth=1)
    
    return Patch(facecolor=color, alpha=0.7, label=f'{label} (TP)'), Patch(facecolor='white', edgecolor=color, label=f'{label} (FP)')

def plotter_comparison(outbreak_country, fpr, retest_delay=3, test_freq=1):
    csv_path = 'AWW_and_ICU/global_model/pgfgleam_code/all_results/global/daily_imports_sensitivity.csv'
    df = pd.read_csv(csv_path)
    df = df[(df['R0'] == 2.0) & (df['generation_time'] == 4.0)]
    gen_time = 10.0
    df_outbreak = df[df['outbreak_country'] == outbreak_country].sort_values('time')
    mu_traj = df_outbreak['daily_detectable_imports'].values

    print(f"Simulating Independent Sliding Window for {outbreak_country}...")
    print(f"  retest_delay={retest_delay}, test_freq={test_freq}")
    # 16% pdet with 50% random sampling
    res = simulate_independent_sliding_window(mu_traj, 0.16*0.5, fpr, retest_delay=retest_delay, test_freq=test_freq)
    perfect = simulate_independent_sliding_window(mu_traj, 0.16*0.5, 0.0, retest_delay=retest_delay, test_freq=test_freq)
    
    fig, axes = plt.subplots(3, 1, figsize=(15, 16))
    if outbreak_country == 'Ireland': 
        xlim = (0, 60) 
    elif outbreak_country == 'Mexico':
        xlim = (0, 80)
    else:
        xlim = (0, 80)
    bins = np.arange(0, 100)
    colors = {1: 'red', 2: 'gray', 3: 'blue'}
    for ax in axes:
        ax.tick_params(axis='both', which='major', labelsize=18)
    
    for i, stage in enumerate([1, 2, 3]):
        ax = axes[i]
        if stage == 1:
            label = 'First'
        elif stage == 2:
            label = 'Two'
        elif stage == 3:
            label = 'Three'
        tp_p, fp_p = plot_split_histogram(ax, res[stage]["confirm_day"], res[stage]["is_true"], 
                                          bins, colors[stage], f"{label} Consecutive", xlim)
        
        ax.hist(perfect[1]["confirm_day"], bins=bins, density=True, histtype='step', 
                color='black', linestyle='--', linewidth=2)
        
        ax.set_xlim(xlim)
        ax.legend(handles=[tp_p, fp_p, plt.Line2D([0], [0], color='black', linestyle='--', label='True First Detection')], 
                  loc='upper left', fontsize=16)
        # ax.set_title(f"{outbreak_country}: {stage}-Consecutive (Independent Window)", fontsize=16)

    # plt.tight_layout()
    plt.savefig(f"figures/{outbreak_country}_delay{retest_delay}_freq{test_freq}_{fpr}.pdf")
    plt.show()

    print(f"\n--- Independent Window Stats: {outbreak_country} ---")
    for s in [1, 2, 3, 4]:
        count = len(res[s]['is_true'])
        if count > 0:
            ppv = np.mean(res[s]['is_true']) * 100
            m_day = np.mean(res[s]['confirm_day'])
            print(f"Stage {s}: N_Samples={count} | PPV={ppv:.1f}% | Avg Confirm Day={m_day:.1f}")

# Examples 

# # Every 3 days
# plotter_comparison('Ireland', 0.04, retest_delay=3, test_freq=3)
# plotter_comparison('Cameroon', 0.04, retest_delay=3, test_freq=3)

# Every day
plotter_comparison('Cambodia', 0.008, retest_delay=3, test_freq=1)
plotter_comparison('Cambodia', 0.04, retest_delay=3, test_freq=1)



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def simulate_independent_sliding_window(mu_trajectory, p, alpha, num_trials=50000, retest_delay=3, test_freq=1, max_consecutive=8):
    """
    Independent Version: 
    Tests only happen on specific days determined by test_freq.
    
    Testing schedule: Samples taken on days 0, test_freq, 2*test_freq, ...
    Results arrive retest_delay days after sampling.
    
    For n consecutive positives starting with sample at day t:
    - 1st positive result: t + retest_delay
    - 2nd positive result: t + test_freq + retest_delay
    - 3rd positive result: t + 2*test_freq + retest_delay
    """
    results = {n: {"is_true": [], "confirm_day": []} for n in range(1, max_consecutive + 1)}

    for _ in range(num_trials):
        found_stages = set()
        
        # Only try starting days that align with testing schedule
        possible_start_days = range(0, len(mu_trajectory), test_freq)
        
        for t in possible_start_days:
            
            # For each N-consecutive target
            for n_conseq in range(1, max_consecutive + 1):
                if n_conseq in found_stages:
                    continue
                
                # Sampling days: t, t+test_freq, t+2*test_freq, ...
                sampling_days = [t + (i * test_freq) for i in range(n_conseq)]
                
                # Check if this specific window stays in bounds
                if sampling_days[-1] >= len(mu_trajectory):
                    continue
                
                # Check if ALL tests in this window are positive
                all_pos = True
                any_infection = False
                
                for sample_day in sampling_days:
                    n_p = np.random.poisson(mu_trajectory[sample_day])
                    prob_tp = 1 - (1 - p)**n_p
                    
                    # Probability of a positive test (TP + FP)
                    prob_pos = prob_tp + alpha * (1 - prob_tp)
                    
                    if np.random.rand() < prob_pos:
                        # If THIS specific test was a TP, record it for truth tracking
                        if n_p > 0:
                            any_infection = True
                    else:
                        all_pos = False
                        break
                
                if all_pos:
                    # The independent criteria (N positives) has been met
                    results[n_conseq]["is_true"].append(any_infection)
                    # Confirmation day = last sampling day + retest_delay
                    results[n_conseq]["confirm_day"].append(sampling_days[-1] + retest_delay)
                    found_stages.add(n_conseq)
            
            if len(found_stages) == max_consecutive:
                break
                
    return results

def generate_independent_heatmaps(outbreak_country, fpr_range, consecutive_range, retest_delay=3, test_freq=1, num_trials=50000):
    csv_path = 'AWW_and_ICU/global_model/pgfgleam_code/all_results/global/daily_imports_sensitivity.csv'
    df = pd.read_csv(csv_path)
    df = df[(df['R0'] == 2.0) & (df['generation_time'] == 4.0)]
    df_outbreak = df[df['outbreak_country'] == outbreak_country].sort_values('time')
    mu_traj = df_outbreak['daily_detectable_imports'].values

    p_val = 0.16 * 0.5  # 16% base detection with 50% sampling
    n_fpr = len(fpr_range)
    n_consec = len(consecutive_range)
    max_conseq = np.max(consecutive_range)
    
    tp_grid = np.zeros((n_consec, n_fpr))
    fp_grid = np.zeros((n_consec, n_fpr)) 
    time_grid = np.zeros((n_consec, n_fpr))
    
    for i, fpr in enumerate(fpr_range):
        print(f"Processing {outbreak_country} | FPR: {fpr*100:.1f}% | test_freq={test_freq}")
        res = simulate_independent_sliding_window(mu_traj, p_val, fpr, num_trials, retest_delay, test_freq, max_conseq)
        
        for j, n in enumerate(consecutive_range):
            is_true = np.array(res[n]["is_true"])
            conf_days = np.array(res[n]["confirm_day"])
            
            if len(is_true) > 0:
                ppv = np.mean(is_true)
                tp_grid[j, i] = ppv * 100
                fp_grid[j, i] = (1 - ppv) * 100
                
                # Mean detection time for true detections
                true_times = conf_days[is_true]
                time_grid[j, i] = np.mean(true_times) if len(true_times) > 0 else np.nan
            else:
                tp_grid[j, i] = fp_grid[j, i] = time_grid[j, i] = np.nan
                
    return tp_grid, fp_grid, time_grid

def plot_heatmaps_for_test_freq(test_freq, retest_delay=3):
    """Generate heatmaps for a specific test_freq value"""
    # Setup ranges
    fpr_range = np.arange(0.01, 0.155, 0.01)
    consecutive_range = np.arange(1, 5)
    
    print(f"\n{'='*60}")
    print(f"Generating heatmaps for test_freq={test_freq}")
    print(f"{'='*60}\n")
    
    # Create x-axis labels from FPR values
    n_fpr = len(fpr_range)
    n_consec = len(consecutive_range)
    xaxis_labels = [f'{fpr*100:.0f}' for fpr in fpr_range]
    
    def format_ax(ax, data, cmap, vmin, vmax, title=''):
        im = ax.imshow(data, aspect='auto', origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
        
        # Colorbar
        cbar = plt.colorbar(im, ax=ax)
        cbar.ax.tick_params(labelsize=18)
        
        # X-axis (FPR)
        # ax.set_xlabel('False Positive Rate (%)', fontsize=20)
        ax.set_xticks(np.arange(n_fpr))
        ax.set_xticklabels(xaxis_labels, rotation=0, ha='center', fontsize=16)
        
        # Y-axis (consecutive tests)
        # ax.set_ylabel('Consecutive Tests', fontsize=20)
        ax.set_yticks(np.arange(n_consec))
        ax.set_yticklabels(consecutive_range, fontsize=18)
        
        # Title
        # if title:
        #     ax.set_title(title, fontsize=22, pad=10)
        
        return im
    
    # Process each country separately
    for country in ['Cambodia']:
        print(f"\nProcessing {country}...")
        
        tp_grid, fp_grid, time_grid = generate_independent_heatmaps(country, 
                                                                     fpr_range, consecutive_range, 
                                                                     retest_delay, test_freq)
        
        # Create 2x1 figure for this country
        fig, axes = plt.subplots(1, 2, figsize=(20, 5))
        
        # Left: FP Rate
        format_ax(axes[0], fp_grid, "RdBu_r", 0, 100, 
                  f'{country} - FP Rate (%) | test_freq={test_freq}d')
        
        # Right: Detection Time
        format_ax(axes[1], time_grid, "RdBu_r", np.nanmin(time_grid), np.nanmax(time_grid), 
                  f'{country} - Detection Time (days) | test_freq={test_freq}d')
        
        plt.tight_layout()
        
        # Create country-specific filename
        country_short = country.replace(' ', '_').replace('Democratic_Republic_of_the_', '')
        filename = f'figures/heatmap_{country_short}_testfreq{test_freq}.pdf'
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"Saved: {filename}")

# Run for both test frequencies
plot_heatmaps_for_test_freq(test_freq=1, retest_delay=3)
# plot_heatmaps_for_test_freq(test_freq=3, retest_delay=3)